In [ ]:
# 1. Fix Protobuf version to prevent compatibility errors
!pip install protobuf==3.20.3

# 2. Install necessary libraries
!pip install --upgrade transformers datasets scikit-learn accelerate

# 3. Suppress logs for a cleaner output
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import warnings
warnings.filterwarnings("ignore")
print("✅ Environment setup completed.")

# 4. Check GPU Status
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Active: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU Not Found! Check Runtime Type.")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import json
from torch.utils.data import WeightedRandomSampler, DataLoader
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

# --- GRANDMASTER HYPERPARAMETERS ---
MODEL_NAME = "microsoft/unixcoder-base"
MAX_LENGTH = 512      # Keeps enough context for code structure analysis
BATCH_SIZE = 128       # High batch size for stable gradients (Try 128 if A100 GPU)
EPOCHS = 5            # 5 Epochs to fully digest the 500k dataset
LEARNING_RATE = 3e-5  # Lower learning rate for precision

# Data Paths (Adjust if needed for Colab)

TRAIN_PATH = "/content/train.parquet"
VAL_PATH = "/content/validation.parquet"
TEST_PATH = "/content/test.parquet"

# !!! CRITICAL: Path to the JSON file from Phase 1
JSON_PATH = "/content/hard_negative_indices.json"

id2label = {
    0: "human", 1: "deepseek", 2: "qwen", 3: "01-ai", 4: "bigcode",
    5: "gemma", 6: "phi", 7: "meta-llama", 8: "ibm-granite", 9: "mistral", 10: "openai"
}
label2id = {v: k for k, v in id2label.items()}

In [ ]:
def prepare_grandmaster_data():
    print("1. Loading FULL Dataset (No Pruning)...")
    # Load the entire dataset
    train_df = pd.read_parquet(TRAIN_PATH).dropna(subset=['code', 'label']).reset_index(drop=True)
    val_df = pd.read_parquet(VAL_PATH).dropna(subset=['code', 'label']).reset_index(drop=True)

    # Use 30k validation samples for robust metrics
    val_df = val_df.sample(n=30000, random_state=42).reset_index(drop=True)

    train_df['label'] = train_df['label'].astype(int)
    val_df['label'] = val_df['label'].astype(int)

    # --- LOAD HARD NEGATIVE MAP ---
    print(f"2. Loading Hard Negative Map from: {JSON_PATH}")
    if not os.path.exists(JSON_PATH):
        raise FileNotFoundError(f"JSON file not found at {JSON_PATH}!")

    with open(JSON_PATH, "r") as f:
        indices_data = json.load(f)
    hard_indices_set = set(indices_data["hard_indices"])

    # --- 3-TIER WEIGHT CALCULATION ---
    print("3. Calculating Grandmaster Weights (3-Tier Strategy)...")
    print("   -> Tier 1: Hard Examples (Human & AI) = 5.0x Priority")
    print("   -> Tier 2: Easy AI Examples (Hidden Gems) = 3.0x Priority")
    print("   -> Tier 3: Easy Humans (Majority) = 1.0x Priority")

    sample_weights = []
    total_samples = len(train_df)
    labels = train_df['label'].values

    for i in range(total_samples):
        # TIER 1: Hard Examples (From JSON)
        if i in hard_indices_set:
            sample_weights.append(5.0)

        # TIER 2: Easy AI Examples (Not in JSON, but label is AI)
        elif labels[i] > 0:
            sample_weights.append(3.0)

        # TIER 3: Easy Humans (Background noise)
        else:
            sample_weights.append(1.0)

    # Compute Class Weights (To assist Focal Loss)
    classes = np.unique(labels)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)

    print(f"   Total Train Size: {total_samples}")
    return train_df, val_df, torch.DoubleTensor(sample_weights), class_weights

# Execute Data Prep
train_df, val_df, sample_weights, class_weights = prepare_grandmaster_data()

1. Loading FULL Dataset (No Pruning)...
2. Loading Hard Negative Map from: /content/hard_negative_indices.json
3. Calculating Grandmaster Weights (3-Tier Strategy)...
   -> Tier 1: Hard Examples (Human & AI) = 5.0x Priority
   -> Tier 2: Easy AI Examples (Hidden Gems) = 3.0x Priority
   -> Tier 3: Easy Humans (Majority) = 1.0x Priority
   Total Train Size: 500000


In [ ]:
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    # We purposefully DO NOT remove comments or whitespaces.
    # Human imperfections (comments, indentation styles) are critical signals.
    return tokenizer(
        examples["code"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

print("Tokenizing dataset (using 4 cores)...")
train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
val_dataset = Dataset.from_pandas(val_df[['code', 'label']])

tokenized_train = train_dataset.map(preprocess_function, batched=True, num_proc=4)
tokenized_val = val_dataset.map(preprocess_function, batched=True, num_proc=4)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_val = tokenized_val.rename_column("label", "labels")
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Tokenizing dataset (using 4 cores)...


Map (num_proc=4):   0%|          | 0/500000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/30000 [00:00<?, ? examples/s]

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        else: return focal_loss

class GrandmasterTrainer(Trainer):
    def __init__(self, *args, class_weights=None, sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.args.device)
        else:
            self.class_weights = None
        self.focal_loss = FocalLoss(alpha=self.class_weights, gamma=2.0)
        self.custom_sampler = sampler

    # Override loss computation to use Focal Loss
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

    # Override dataloader to inject our 3-Tier Sampler
    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=self.custom_sampler,
            collate_fn=self.data_collator,
            num_workers=4,
            pin_memory=True
        )

In [ ]:
# Load Fresh Model (Reset weights)
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=11,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1_macro': f1}

# --- SAMPLER CONFIGURATION ---
# We process the FULL dataset size per epoch.
# However, due to weights, Hard/AI examples will appear 3x-5x more often.
samples_per_epoch = len(train_df)
print(f"⚡ GRANDMASTER PLAN: Processing {samples_per_epoch} samples per epoch.")

smart_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=samples_per_epoch,
    replacement=True
)

training_args = TrainingArguments(
    output_dir="./unixcoder_grandmaster_phase3",
    num_train_epochs=EPOCHS,           # 5 Epochs
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,       # 2e-5
    weight_decay=0.01,

    # Save/Eval every 2000 steps to monitor progress
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True, # Enable Mixed Precision for speed
    report_to="none"
)

trainer = GrandmasterTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    sampler=smart_sampler
)

print("🚀 STARTING GRANDMASTER TRAINING...")
trainer.train()

# Save the final champion model
trainer.save_model("./final_grandmaster_model")
tokenizer.save_pretrained("./final_grandmaster_model")
print("✅ Training Completed Successfully.")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


⚡ GRANDMASTER PLAN: Processing 500000 samples per epoch.
🚀 STARTING GRANDMASTER TRAINING...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,1.772700,1.143864,0.159300,0.182133
4000,1.356500,1.058105,0.311400,0.263847
6000,1.047500,1.097600,0.335367,0.278461
8000,0.751900,1.196673,0.487467,0.297942
10000,0.564500,1.395293,0.494633,0.325742
12000,0.400900,1.468089,0.529033,0.330544
14000,0.284500,1.502963,0.590067,0.355892
16000,0.195500,1.664808,0.612700,0.382099
18000,0.145500,1.718634,0.681067,0.388731


✅ Training Completed Successfully.


In [ ]:
# 1. Generate Submission File
print("Loading Test Data...")
test_df = pd.read_parquet(TEST_PATH)
test_dataset = Dataset.from_pandas(test_df)
tokenized_test = test_dataset.map(preprocess_function, batched=True, num_proc=4)

print("Running Prediction on Test Set...")
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=1)

submission = pd.DataFrame()
# Handle ID column variations
if 'ID' in test_df.columns: submission['ID'] = test_df['ID']
elif 'id' in test_df.columns: submission['ID'] = test_df['id']
else: submission['ID'] = test_df.index
submission['label'] = preds
submission.to_csv('submission_grandmaster.csv', index=False)
print("✅ submission_grandmaster.csv generated!")

# 2. Detailed Performance Report
print("\n🔍 FINAL REPORT (Checking AI vs Human Balance):")
from sklearn.metrics import classification_report
val_preds = trainer.predict(tokenized_val)
print(classification_report(val_preds.label_ids, np.argmax(val_preds.predictions, axis=1), target_names=[id2label[i] for i in range(11)], digits=4))

Loading Test Data...


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Running Prediction on Test Set...


✅ submission_grandmaster.csv generated!

🔍 FINAL REPORT (Checking AI vs Human Balance):


              precision    recall  f1-score   support

       human     0.9987    0.6914    0.8171     26521
    deepseek     0.0591    0.5976    0.1076       246
        qwen     0.1122    0.4832    0.1821       507
       01-ai     0.2293    0.5211    0.3185       213
     bigcode     0.3279    0.6107    0.4267       131
       gemma     0.3594    0.7667    0.4894       120
         phi     0.4880    0.5000    0.4939       324
  meta-llama     0.0887    0.4113    0.1459       496
 ibm-granite     0.2274    0.7860    0.3528       500
     mistral     0.1658    0.4480    0.2420       279
      openai     0.6165    0.8100    0.7001       663

    accuracy                         0.6811     30000
   macro avg     0.3339    0.6024    0.3887     30000
weighted avg     0.9154    0.6811    0.7637     30000



In [ ]:
print(f"📄 Submission length: {len(submission)}")

📄 Submission length: 1000


In [ ]:
# --- PHASE 4: COOL-DOWN STABILIZATION (CORRECTED) ---

print("❄️ STARTING PHASE 4: COOL-DOWN STABILIZATION...")

# 1. Load the Grandmaster Model
# We continue from where Phase 3 left off
model_path = "./final_grandmaster_model"
print(f"   Loading trained weights from: {model_path}")

model = RobertaForSequenceClassification.from_pretrained(
    model_path,
    num_labels=11,
    id2label=id2label,
    label2id=label2id
)

# 2. Data Strategy: Natural Distribution
print("   Switching to NATURAL distribution (No Weighted Sampler)...")
# We want the model to see the real ratio (400k Humans vs 50k AI)
# to fix the "paranoia" and improve Human Recall.

# 3. Training Arguments (Very Gentle)
training_args_cooling = TrainingArguments(
    output_dir="./unixcoder_phase4_stable",
    num_train_epochs=1,                # 1 Epoch is enough to settle
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=5e-6,                # !!! VERY LOW LR (Prevents forgetting AI knowledge)
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    report_to="none"
)

# --- CUSTOM TRAINER (FIXED) ---
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # !!! FIX: Convert NumPy array to PyTorch Tensor before moving to GPU !!!
        if isinstance(class_weights, torch.Tensor):
            weight_tensor = class_weights.to(model.device)
        else:
            weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(model.device)

        # We still use Class Weights to protect the weak AI classes (Gemma/BigCode)
        loss_fct = nn.CrossEntropyLoss(weight=weight_tensor)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Initialize Trainer
trainer_cooling = WeightedTrainer(
    model=model,
    args=training_args_cooling,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

print("🚀 STARTING COOL-DOWN TRAINING (1 Epoch)...")
trainer_cooling.train()

# Save Final Model
trainer_cooling.save_model("./final_phase4_model")
tokenizer.save_pretrained("./final_phase4_model")
print("✅ Phase 4 Completed.")

# --- FINAL REPORT ---
print("\n🔍 PHASE 4 REPORT (Did Human Recall improve?):")
val_preds = trainer_cooling.predict(tokenized_val)
print(classification_report(val_preds.label_ids, np.argmax(val_preds.predictions, axis=1), target_names=[id2label[i] for i in range(11)], digits=4))

# Generate Submission
print("Generating Submission...")
test_df = pd.read_parquet(TEST_PATH)
test_dataset = Dataset.from_pandas(test_df)
tokenized_test = test_dataset.map(preprocess_function, batched=True, num_proc=4)
predictions = trainer_cooling.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=1)

submission = pd.DataFrame()
if 'ID' in test_df.columns: submission['ID'] = test_df['ID']
elif 'id' in test_df.columns: submission['ID'] = test_df['id']
else: submission['ID'] = test_df.index
submission['label'] = preds
submission.to_csv('submission_phase4.csv', index=False)
print("✅ submission_phase4.csv ready!")

❄️ STARTING PHASE 4: COOL-DOWN STABILIZATION...
   Loading trained weights from: ./final_grandmaster_model
   Switching to NATURAL distribution (No Weighted Sampler)...
🚀 STARTING COOL-DOWN TRAINING (1 Epoch)...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
1000,0.175200,1.917648,0.900633,0.529629
2000,0.190600,2.047261,0.913933,0.547013
3000,0.181200,2.043508,0.913567,0.543874
4000,0.179000,2.112447,0.907633,0.542828
5000,0.201200,2.221315,0.911133,0.541968
6000,0.173200,2.191079,0.916067,0.550587
7000,0.167500,2.194920,0.915133,0.550754


✅ Phase 4 Completed.

🔍 PHASE 4 REPORT (Stabilization Check):


              precision    recall  f1-score   support

       human     0.9946    0.9578    0.9759     26521
    deepseek     0.2691    0.5447    0.3602       246
        qwen     0.3214    0.4970    0.3904       507
       01-ai     0.4206    0.4977    0.4559       213
     bigcode     0.5469    0.5344    0.5405       131
       gemma     0.5380    0.7667    0.6323       120
         phi     0.5763    0.5247    0.5493       324
  meta-llama     0.3752    0.4274    0.3996       496
 ibm-granite     0.5746    0.7860    0.6639       500
     mistral     0.3000    0.4086    0.3460       279
      openai     0.7236    0.7662    0.7443       663

    accuracy                         0.9151     30000
   macro avg     0.5128    0.6101    0.5508     30000
weighted avg     0.9352    0.9151    0.9237     30000

Generating Submission...


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ submission_phase4.csv ready!
